# Sesión 12 - Lab 2: Control de acceso, auditoría y trazabilidad

Este notebook reutiliza tablas reales del catálogo `dbassociate` (Sesiones 08 y 09) para GRANT/REVOKE, y crea un único schema descartable propio (`dbassociate.sesion12_lab2_demo`) para el paso de ownership/MANAGE, independiente del schema del Lab 1.

Prerrequisito: los tres grupos de cuenta `curso_analistas`, `curso_ingenieria` y `curso_auditoria` ya creados desde el account console, sin necesidad de miembros. El notebook verifica el estado de los permisos otorgados a esos grupos, no requiere iniciar sesión como ellos.

## Verificación del entorno

In [ ]:
display(spark.sql("SHOW SCHEMAS IN dbassociate"))
display(spark.sql("SELECT current_user() AS usuario_actual"))

## Lab 2A — Estado inicial: owner y permisos ya otorgados

In [ ]:
display(spark.sql("SHOW GRANTS ON CATALOG dbassociate"))
display(spark.sql("SHOW GRANTS ON SCHEMA dbassociate.silver"))
display(spark.sql("SHOW GRANTS ON TABLE dbassociate.silver.clientes"))

In [ ]:
display(spark.sql("DESCRIBE SCHEMA EXTENDED dbassociate.silver"))

Buscá la fila `Owner` en la salida de arriba. Por defecto es quien creó el schema; se transfiere con `ALTER SCHEMA ... OWNER TO`, que practicamos más abajo en el Lab 2H sobre un schema descartable, no sobre este.

## Lab 2B — GRANT a nivel catalog: alcance amplio, herencia automática

In [ ]:
spark.sql("GRANT USE CATALOG ON CATALOG dbassociate TO `curso_analistas`")
spark.sql("GRANT USE SCHEMA, SELECT ON CATALOG dbassociate TO `curso_analistas`")

display(spark.sql("""
SELECT table_schema, table_name, privilege_type, inherited_from
FROM system.information_schema.table_privileges
WHERE table_catalog = 'dbassociate' AND grantee = 'curso_analistas'
ORDER BY table_schema, table_name
"""))

`inherited_from` debería mostrar `dbassociate` en cada fila, para cada tabla de cada schema: un solo `GRANT` a nivel catalog se propaga a todo lo que hay debajo, actual y futuro.

## Lab 2C — GRANT a nivel schema: alcance acotado a un solo schema

In [ ]:
spark.sql("GRANT USE CATALOG ON CATALOG dbassociate TO `curso_ingenieria`")
spark.sql("GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA dbassociate.silver TO `curso_ingenieria`")

display(spark.sql("""
SELECT table_schema, table_name, privilege_type, inherited_from
FROM system.information_schema.table_privileges
WHERE table_catalog = 'dbassociate' AND grantee = 'curso_ingenieria'
ORDER BY table_schema, table_name
"""))

A diferencia del Lab 2B, acá solo aparecen filas de `silver`: el `GRANT` quedó acotado a ese schema, `bronze` y `gold` no se ven afectados.

## Lab 2D — GRANT a nivel tabla: el más fino, sin herencia hacia hermanos

In [ ]:
spark.sql("GRANT BROWSE ON CATALOG dbassociate TO `curso_auditoria`")
spark.sql("GRANT USE SCHEMA ON SCHEMA dbassociate.silver TO `curso_auditoria`")
spark.sql("GRANT READ METADATA ON TABLE dbassociate.silver.clientes TO `curso_auditoria`")

display(spark.sql("""
SELECT table_schema, table_name, privilege_type, inherited_from
FROM system.information_schema.table_privileges
WHERE table_catalog = 'dbassociate' AND grantee = 'curso_auditoria'
ORDER BY table_schema, table_name
"""))

`BROWSE` deja a `curso_auditoria` navegar la estructura de todo `dbassociate` sin necesitar `USE CATALOG`/`USE SCHEMA` para eso; `READ METADATA` sobre la tabla puntual deja ver su definición (columnas, comentarios) sin darle `SELECT` sobre los datos. Es el rol de auditoría: ve la forma de las cosas, no el contenido.

## Lab 2E — GRANT ON METASTORE: el nivel que no hereda (salvo una excepción)

In [ ]:
spark.sql("GRANT CREATE CATALOG ON METASTORE TO `curso_ingenieria`")

display(spark.sql("SHOW GRANTS `curso_ingenieria` ON METASTORE"))
display(spark.sql("SHOW GRANTS `curso_ingenieria` ON CATALOG dbassociate"))

`CREATE CATALOG` aparece en el primer `SHOW GRANTS` (a nivel metastore) pero no en el segundo (a nivel catalog): no heredó hacia abajo. Ahora la excepción, con `READ METADATA`:

In [ ]:
spark.sql("GRANT READ METADATA ON METASTORE TO `curso_auditoria`")

display(spark.sql("""
SELECT table_catalog, table_schema, table_name, privilege_type, inherited_from
FROM system.information_schema.table_privileges
WHERE grantee = 'curso_auditoria' AND privilege_type = 'READ_METADATA'
ORDER BY table_catalog, table_schema, table_name
"""))

Acá sí deberían aparecer filas de varios catálogos, no solo `dbassociate`: `READ METADATA` es la única excepción documentada donde un privilegio de metastore sí se propaga hacia todos los objetos por debajo.

## Lab 2F — REVOKE: deshacer un grant, e idempotencia

In [ ]:
spark.sql("REVOKE USE SCHEMA, SELECT ON CATALOG dbassociate FROM `curso_analistas`")

display(spark.sql("""
SELECT table_schema, table_name, privilege_type, inherited_from
FROM system.information_schema.table_privileges
WHERE table_catalog = 'dbassociate' AND grantee = 'curso_analistas'
"""))

spark.sql("REVOKE SELECT ON TABLE dbassociate.silver.clientes FROM `curso_analistas`")
print("El REVOKE anterior corrió sin error, aunque curso_analistas ya no tenía ese privilegio: REVOKE es idempotente.")

## Lab 2G — DENY: por qué no hay ninguna celda ejecutable acá

Databricks SQL sí tiene una sentencia `DENY`, con esta sintaxis:

```sql
DENY <privilegio> ON <tipo> <nombre> TO <principal>
```

Pero su propia documentación aclara que **solo aplica al catálogo legacy `hive_metastore` y sus objetos**. Contra un securable de Unity Catalog (como cualquiera de los que venimos usando en este notebook), `DENY` no está soportado.

El motivo no es una limitación de implementación: es una decisión de diseño. El modelo de privilegios de Unity Catalog se construye sobre el principio de mínimo privilegio: todo lo que no está otorgado queda implícitamente denegado. No hace falta un verbo explícito para negar algo que, por default, ya no está permitido.

Para quitar un acceso que sí se otorgó, el mecanismo real es el que ya practicamos en el Lab 2F: `REVOKE`. Un modelo de acceso en Unity Catalog se diseña siempre en positivo (qué se otorga a cada principal), nunca listando excepciones negativas.

## Lab 2H — Ownership y MANAGE, sobre un schema descartable propio

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbassociate.sesion12_lab2_demo")
spark.sql("CREATE OR REPLACE TABLE dbassociate.sesion12_lab2_demo.demo_ownership AS SELECT 1 AS id")

display(spark.sql("DESCRIBE SCHEMA EXTENDED dbassociate.sesion12_lab2_demo"))

In [ ]:
spark.sql("ALTER SCHEMA dbassociate.sesion12_lab2_demo OWNER TO `curso_ingenieria`")
spark.sql("GRANT MANAGE ON SCHEMA dbassociate.sesion12_lab2_demo TO `curso_auditoria`")

display(spark.sql("DESCRIBE SCHEMA EXTENDED dbassociate.sesion12_lab2_demo"))

`curso_ingenieria` ahora es owner del schema, y por eso tiene control automático sobre `demo_ownership` aunque no sea su owner directo. `curso_auditoria` recibió `MANAGE`: puede otorgar y revocar privilegios sobre el schema sin ser su owner. Databricks recomienda este patrón (ownership y `MANAGE` a grupos, no a personas) precisamente para no depender de quién creó cada objeto.

## Lab 2I — Equivalencia por UI

Todo lo que hicimos por SQL en los Labs 2B-2H tiene el mismo resultado desde **Catalog Explorer**:

1. **Catalog** → seleccionar el catalog, schema o tabla.
2. Pestaña **Permissions** → **Grant** para otorgar, o seleccionar un privilegio ya otorgado y **Revoke** para quitarlo.
3. La pestaña **Owner**, en la vista de detalle del objeto, permite transferir ownership sin escribir `ALTER ... OWNER TO`.

No hay ninguna opción de "Deny" en esa UI: la interfaz refleja el mismo modelo que la sintaxis SQL.

## Lab 2J — Lineage: de dónde viene y a dónde va cada tabla

In [ ]:
display(spark.sql("""
SELECT source_table_full_name, source_type, target_table_full_name, target_type, event_time
FROM system.access.table_lineage
WHERE target_table_full_name = 'dbassociate.silver.clientes'
   OR source_table_full_name = 'dbassociate.silver.clientes'
ORDER BY event_time DESC
LIMIT 50
"""))

In [ ]:
display(spark.sql("""
SELECT source_table_full_name, source_column_name, target_table_full_name, target_column_name
FROM system.access.column_lineage
WHERE target_table_full_name = 'dbassociate.gold.clientes_resumen_mensual'
ORDER BY event_time DESC
LIMIT 50
"""))

Si estas dos consultas devuelven filas, deberías ver la cadena real de la Sesión 09: `dbassociate.bronze.clientes_raw` → `dbassociate.silver.clientes` → `dbassociate.gold.clientes_resumen_mensual`, capturada por Unity Catalog en el momento en que ese pipeline corrió, no reconstruida a mano. Lo mismo se ve, en formato gráfico, en la pestaña **Lineage** de Catalog Explorer sobre cualquiera de esas tablas.

## Lab 2K — Auditoría: quién cambió qué permiso, y cuándo

In [ ]:
display(spark.sql("""
SELECT event_time, user_identity.email AS usuario, action_name,
       request_params.securable_type AS tipo_objeto,
       request_params.securable_full_name AS objeto,
       request_params.changes AS cambios
FROM system.access.audit
WHERE event_date >= current_date() - INTERVAL 30 DAYS
  AND service_name = 'unityCatalog'
  AND action_name = 'updatePermissions'
ORDER BY event_time DESC
LIMIT 50
"""))

Si no ves ahí los `GRANT`/`REVOKE` que acabamos de correr en este mismo notebook, no es un error: `system.access.audit` no es inmediato, puede tardar en reflejar eventos recientes. Lo que sí deberías ver es historial de sesiones anteriores del curso: creaciones de tabla, cambios de permisos previos, etc.

In [ ]:
display(spark.sql("""
SELECT event_time, user_identity.email AS usuario, action_name,
       request_params.full_name_arg AS objeto
FROM system.access.audit
WHERE event_date >= current_date() - INTERVAL 30 DAYS
  AND service_name = 'unityCatalog'
  AND action_name IN ('createTable', 'deleteTable')
  AND request_params.full_name_arg LIKE 'dbassociate.sesion12%'
ORDER BY event_time DESC
LIMIT 50
"""))

## Limpieza

In [ ]:
for privilegio, tipo, objeto, principal in [
    ("USE CATALOG", "CATALOG", "dbassociate", "curso_analistas"),
    ("USE CATALOG", "CATALOG", "dbassociate", "curso_ingenieria"),
    ("USE SCHEMA, SELECT, MODIFY", "SCHEMA", "dbassociate.silver", "curso_ingenieria"),
    ("BROWSE", "CATALOG", "dbassociate", "curso_auditoria"),
    ("USE SCHEMA", "SCHEMA", "dbassociate.silver", "curso_auditoria"),
    ("READ METADATA", "TABLE", "dbassociate.silver.clientes", "curso_auditoria"),
]:
    spark.sql(f"REVOKE {privilegio} ON {tipo} {objeto} FROM `{principal}`")

spark.sql("REVOKE CREATE CATALOG ON METASTORE FROM `curso_ingenieria`")
spark.sql("REVOKE READ METADATA ON METASTORE FROM `curso_auditoria`")

spark.sql("ALTER SCHEMA dbassociate.sesion12_lab2_demo OWNER TO `" + spark.sql("SELECT current_user() AS u").first()["u"] + "`")
spark.sql("REVOKE MANAGE ON SCHEMA dbassociate.sesion12_lab2_demo FROM `curso_auditoria`")
spark.sql("DROP TABLE IF EXISTS dbassociate.sesion12_lab2_demo.demo_ownership")
spark.sql("DROP SCHEMA IF EXISTS dbassociate.sesion12_lab2_demo")

print("Grants revocados y schema descartable eliminado. dbassociate.bronze/silver/gold quedan exactamente como estaban antes de este notebook.")